# Step 06.3 — Next-Token Shift：一段文本怎样自动变成监督学习题

前面已经完成：

```text
text -> token IDs -> embedding vectors
```

但语言模型还缺一个最关键的问题：**target 从哪里来？**

答案非常巧妙：对自回归语言模型来说，target 就藏在原文本的“下一位”里。

这一节完整解决 5 个问题：

1. next-token prediction 到底在预测什么；
2. 为什么一段 `T+1` token 可以切成长度都为 `T` 的 input / target；
3. 为什么一个长度为 `T` 的 block 实际提供 `T` 个监督信号；
4. Batch 后 `(B,T)` 的 shape 怎样形成；
5. target 明明来自未来 token，为什么训练不算作弊，以及 causal mask 将来为什么必须存在。

## 1. Next-token prediction：目标不是“复原输入”，而是预测右边那个 token

先看最简单字符串：

```text
hello
```

语言模型希望学习的是这样的条件关系：

```text
看到 "h"    -> 下一个可能是 "e"
看到 "he"   -> 下一个可能是 "l"
看到 "hel"  -> 下一个可能是 "l"
看到 "hell" -> 下一个可能是 "o"
```

因此它不是学习：

```text
h -> h
e -> e
l -> l
```

而是在学习：

```text
当前位置已有的上下文 -> 下一 token
```

这就是 **autoregressive next-token prediction（自回归下一 token 预测）**。

In [ ]:
text = "hello"

chars = sorted(set(text))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

def encode(s):
    return [stoi[ch] for ch in s]

def decode(ids):
    return "".join(itos[int(i)] for i in ids)

ids = encode(text)

print("chars:", chars)
print("text :", text)
print("ids  :", ids)

注意一个很重要的思想：

> **语言模型的数据标签不需要人工逐字标注。原始文本本身就同时提供 input 和 target。**

只要把同一串 token 错开一位即可。

## 2. `T+1 -> T + T`：input / target 为什么只错开一位

对：

```text
hello
```

取：

```text
input  x = hell
target y = ello
```

写成位置对齐：

```text
position:  0   1   2   3
input x :  h   e   l   l
target y:  e   l   l   o
            ↑   ↑   ↑   ↑
           next token target
```

所以如果训练输入长度要是 `T=4`，原始 chunk 必须先拿 `T+1=5` 个 token。

代码不是循环移动（circular roll），而是普通切片：

```python
x = chunk[:-1]
y = chunk[1:]
```

这里**不会**把最后一个 token 绕回开头。

In [ ]:
import mlx.core as mx

chunk = mx.array(ids)            # 5 tokens: hello
x = chunk[:-1]                  # first 4: hell
y = chunk[1:]                   # last 4:  ello

print("chunk IDs:", chunk, "shape =", chunk.shape)
print("x IDs    :", x, "shape =", x.shape, "text =", decode(x.tolist()))
print("y IDs    :", y, "shape =", y.shape, "text =", decode(y.tolist()))

### 画一张 input-target 对齐图

这一类“序列对应关系”很适合画图，而不是只盯数组。

这里开始使用 Matplotlib 更常见的 **object-oriented** 写法：

```python
fig, ax = plt.subplots(...)
```

`fig` 是整张画布，`ax` 是具体坐标区域。后面图变复杂时，这种写法比一直调用 `plt.xxx()` 更容易控制。

In [ ]:
import matplotlib.pyplot as plt

x_chars = list("hell")
y_chars = list("ello")
positions = list(range(len(x_chars)))

fig, ax = plt.subplots(figsize=(8, 3))

ax.scatter(positions, [1] * len(positions), s=80)
ax.scatter(positions, [0] * len(positions), s=80)

for t, (xc, yc) in enumerate(zip(x_chars, y_chars)):
    ax.annotate(xc, (t, 1), xytext=(0, 10), textcoords="offset points", ha="center")
    ax.annotate(yc, (t, 0), xytext=(0, -18), textcoords="offset points", ha="center")
    ax.annotate("", xy=(t, 0.15), xytext=(t, 0.85), arrowprops={"arrowstyle": "->"})

ax.text(-0.55, 1, "input x", va="center")
ax.text(-0.55, 0, "target y", va="center")
ax.set_xticks(positions)
ax.set_xlabel("position t")
ax.set_yticks([])
ax.set_ylim(-0.45, 1.45)
ax.set_title("Next-token supervision: each position predicts the token one step to the right")
plt.show()

图里的竖箭头表示：**模型在 position `t` 的输出，要和 `y[t]` 比较。**

但还要马上补一个更精确的说法：position `t` 的预测最终不是只看 `x[t]`，而是应该使用到当前为止的整个前缀上下文。下一小节专门讲这个关键点。

## 3. 为什么一个长度为 `T` 的 block 能产生 `T` 个训练题？

对于：

```text
x = h e l l
y = e l l o
```

不能简单理解成 4 个彼此独立的：

```text
h -> e
e -> l
l -> l
l -> o
```

真正的 causal language model 语义是：

```text
position 0: context "h"    -> target "e"
position 1: context "he"   -> target "l"
position 2: context "hel"  -> target "l"
position 3: context "hell" -> target "o"
```

也就是说，同一个长度为 4 的 block，同时贡献了 4 个不同 context length 的监督信号。

这非常高效：模型一次 forward 可以为所有位置都输出预测，而不需要把 `h`、`he`、`hel`、`hell` 分别跑四次模型。

In [ ]:
x_text = "hell"
y_text = "ello"

for t in range(len(x_text)):
    context = x_text[:t + 1]
    target_char = y_text[t]
    print(f"position {t}: context={context!r:<8} -> target={target_char!r}")

这也解释了以后模型输出为什么不是只要最后一个位置。

训练时会得到类似：

```text
logits.shape = (B, T, V)
```

每一个 `(b,t)` 位置都有一个长度为 `V` 的 next-token 预测分数向量，并与：

```text
targets.shape = (B,T)
```

对应位置的正确 token ID 计算 loss。

Cross Entropy 要到 Step 17 才正式推导，这里先把 shape 和监督关系建立起来。

## 4. 加入 Batch：`(B,T+1) -> X(B,T), Y(B,T)`

真实训练不会只拿一个文本窗口。

我们从语料：

```text
hello lulu\n
```

取两个长度为 5 的 chunk：

```text
chunk 0 = hello
chunk 1 = lulu\n
```

于是原始 batch chunk 的 shape 是：

```text
(B,T+1) = (2,5)
```

统一切片后：

```text
X = batch[:, :-1] -> (2,4)
Y = batch[:, 1:]  -> (2,4)
```

In [ ]:
corpus = "hello lulu\n"
chars2 = sorted(set(corpus))
stoi2 = {ch: i for i, ch in enumerate(chars2)}
itos2 = {i: ch for i, ch in enumerate(chars2)}

def encode2(s):
    return [stoi2[ch] for ch in s]

def decode2(ids):
    return "".join(itos2[int(i)] for i in ids)

data = mx.array(encode2(corpus))

batch_chunks = mx.stack([
    data[0:5],      # hello
    data[6:11],     # lulu\n
])

X = batch_chunks[:, :-1]
Y = batch_chunks[:, 1:]

print("batch_chunks shape:", batch_chunks.shape)
print("X shape           :", X.shape)
print("Y shape           :", Y.shape)

for b in range(X.shape[0]):
    print(f"batch {b}: X={decode2(X[b].tolist())!r}  Y={decode2(Y[b].tolist())!r}")

这里把你前面学过的 shape 链起来：

```text
X token IDs        : (B,T)
token embedding    : (B,T,C)
model output logits: (B,T,V)
targets Y          : (B,T)
```

注意 `Y` 不需要 embedding 后再作为分类标签使用；它保留整数 token ID，告诉 loss：每个位置正确类别是哪一个。

## 5. Target 来自未来，为什么训练不算作弊？——Teacher Forcing 与 Causal Constraint

这是 next-token training 最容易产生误解的地方。

我们确实把整段：

```text
x = h e l l
y = e l l o
```

同时放在内存里，但 **Y 不作为模型当前位置的输入**。

训练逻辑是：

```text
X -> model -> predictions
Y ----------> loss comparison
```

Y 的职责是评分，不是告诉模型答案。

### 但是还有一个更隐蔽的泄漏问题

观察：

```text
X = h e l l
Y = e l l o
```

position 0 的正确答案是 `e`，而 `e` 恰好就出现在 `X` 的 position 1。

如果未来的 Transformer 在 position 0 可以随便看到 position 1、2、3，那么它就能直接偷看答案。

所以 causal language model 必须满足：

```text
position t 只能看到 positions <= t
不能看到 positions > t
```

以后我们会用 **causal mask** 强制实现这个约束。

这说明：

> `shifted targets` 定义“要预测谁”；`causal mask` 定义“预测时允许看谁”。

两者是不同问题，但必须配合。

### Teacher Forcing 是什么？

训练时，position 1 的上下文使用真实文本里的前一个 token `e`，而不是先让模型生成一个 token 再继续。

这种使用 ground-truth previous tokens 作为训练输入的思想通常称为 **teacher forcing**。

推理时则不同：没有未来真实文本可用，模型必须：

```text
输入已有 context
-> 预测 next token
-> 把预测 token 接回 context
-> 再预测下一个
```

这就是 autoregressive generation，后面 Step 21 会完整实现。

### 6.3 最终结论

- next-token prediction 的 target 就是原序列向右一位的 token；
- 要得到长度 `T` 的 X/Y，需要先取 `T+1` 个原始 token；
- `X = chunk[:-1]`，`Y = chunk[1:]`，不是 circular roll；
- 一个长度为 `T` 的 causal block 同时提供 `T` 个不同前缀长度的监督信号；
- Batch 后从 `(B,T+1)` 切成 `X(B,T)` 和 `Y(B,T)`；
- X 之后会进入 embedding/model，Y 保持整数类别 ID 用于 loss；
- target shift 决定预测目标，causal mask 防止未来信息泄漏；
- 下一节 6.4：为什么不能把整本书一次塞进模型，以及 `context window / block_size / T` 到底限制了什么。